In [1]:
import pandas as pd
from migration.datasets import TensorflowEncodedBatchedInferenceDatasetBuilder
from pathlib import Path

2025-07-24 13:01:26.814447: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753362086.831890 1965155 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753362086.836979 1965155 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753362086.852008 1965155 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753362086.852027 1965155 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753362086.852029 1965155 computation_placer.cc:177] computation placer alr

In [2]:
from migration.config import DatasetConfig, TrainingConfig
def _get_config() -> TrainingConfig:
    return TrainingConfig(
    dataset=DatasetConfig(
        training_parquet='../new_data/ais_train_arrow.parquet',
        validation_parquet='../new_data/ais_val_arrow.parquet',
        test_parquet='../new_data/ais_test.parquet',
        mean_pickle='../data/ct_2017010203_10_20/mean.pkl',
        shuffle=False,
        val_size=15813 // 32,
        training_size=73795 // 32,
    ), epochs=9999)
cfg = _get_config()

In [3]:

def get_pandas_generator(parquet : Path):
    def get_in_memory_dataset_generator():
        _ds = pd.read_parquet(parquet, columns=['latitude', 'longitude', 'sog', 'cog'])
        _idxs = _ds.index.unique()
        for idx in _idxs:
            track = _ds.loc[idx].values
            yield idx, track.reshape((-1, 4))
    return get_in_memory_dataset_generator

In [13]:
_ds = pd.read_parquet('../new_data/ais_test.parquet', columns=['latitude', 'longitude', 'sog', 'cog'])

In [15]:
len(_ds)

989426

In [4]:
cfg_ds = cfg.dataset
train_generator = get_pandas_generator(cfg_ds.training_parquet)
val_generator = get_pandas_generator(cfg_ds.validation_parquet)
train = TensorflowEncodedBatchedInferenceDatasetBuilder(
        track_with_id_generator=train_generator,
        batch_size=cfg_ds.batch_size,
        lat_bins=cfg_ds.encoding_bins.lat,
        lon_bins=cfg_ds.encoding_bins.lon,
        sog_bins=cfg_ds.encoding_bins.sog,
        cog_bins=cfg_ds.encoding_bins.cog,
        shuffle=cfg_ds.shuffle,
        repeat=True
    ).build()

I0000 00:00:1753362089.304464 1965155 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 41768 MB memory:  -> device: 0, name: NVIDIA L40S-48Q, pci bus id: 0000:00:05.0, compute capability: 8.9


In [8]:
it = iter(train)

In [16]:
a = np.zeros(6)

In [18]:
a[0:5] = range(5)

In [ ]:
a

array([0., 1., 2., 3., 4., 0.])

: 

In [9]:
next(it)

(<tf.Tensor: shape=(32,), dtype=uint16, numpy=
 array([339, 342, 345, 354, 406, 424, 428, 439, 448, 462, 467, 469, 480,
        482, 476, 484, 535, 491, 541, 543, 546, 596, 582, 598, 603, 620,
        614, 622, 624, 625, 643, 668], dtype=uint16)>,
 <tf.Tensor: shape=(120, 32, 702), dtype=float32, numpy=
 array([[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 1., 0., ..., 1., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],
 
        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 1., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
       

In [5]:
sub =  ['test', 'train', 'val'][0]
parquet = f'../new_data/ais_{sub}_arrow.parquet'
_ds = pd.read_parquet(parquet, columns=['latitude', 'longitude', 't'])

In [12]:
import numpy as np
np.mean(np.array([[2,3,5], 
                  [2,2,2]]), axis=1)

array([3.33333333, 2.        ])

In [6]:
_ds

,latitude,longitude,t
track_id,,,
239,0.563266,0.291609,0
239,0.562646,0.287027,1
239,0.562080,0.282530,2
239,0.562011,0.278010,3
239,0.562501,0.273443,4
...,...,...,...
511825,0.632873,0.022723,57
511825,0.633535,0.015191,58
511825,0.635315,0.010628,59


In [7]:
_ds

,latitude,longitude,t
track_id,,,
239,0.563266,0.291609,0
239,0.562646,0.287027,1
239,0.562080,0.282530,2
239,0.562011,0.278010,3
239,0.562501,0.273443,4
...,...,...,...
511825,0.632873,0.022723,57
511825,0.633535,0.015191,58
511825,0.635315,0.010628,59
